# AdaptiveSRE — Colab Training Notebook

**Runtime → Change runtime type → T4 GPU** before running.

| Cell | Purpose | Time |
|------|---------|------|
| 1 | Install deps (unsloth from git) | ~3 min |
| 2 | Clone repo, verify GPU, start services | ~1 min |
| 3 | Smoke test (5 episodes) | ~5 min |
| 4 | Full GRPO training (200 episodes) | ~1-2 hr |
| 5 | Evaluate & plot | ~15 min |

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
# IMPORTANT: Do NOT pip install torch — Colab has CUDA torch
# Unsloth MUST come from git with [colab-new] extras
# ============================================================

!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps unsloth_zoo
!pip install -q xformers trl peft accelerate bitsandbytes triton
!pip install -q datasets sentencepiece protobuf
!pip install -q fastapi uvicorn pydantic httpx matplotlib

# Verify core imports
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

from unsloth import FastLanguageModel
print("Unsloth OK")

from trl import GRPOConfig, GRPOTrainer
print("TRL OK")

print("\n=== Cell 1 PASSED ===")

In [ ]:
# ============================================================
# Cell 2: Clone Repo & Start Mock Services
# ============================================================
import os, subprocess, time

# Clone if not already present
if not os.path.isdir("Adaptive-SRE"):
    !git clone https://github.com/ashifsekh/Adaptive-SRE.git

os.chdir("/content/Adaptive-SRE")
print(f"Working dir: {os.getcwd()}")

# Ensure __init__.py exists for mock_services sub-packages
for d in ["mock_services", "mock_services/db", "mock_services/auth",
          "mock_services/payment", "mock_services/cache", "mock_services/notification"]:
    init = os.path.join(d, "__init__.py")
    if not os.path.exists(init):
        open(init, "w").close()
        print(f"  Created {init}")

# Kill any leftover uvicorn from previous runs
!pkill -f uvicorn 2>/dev/null || true
time.sleep(1)

# Start 5 mock services + main server in background
services = [
    ("mock_services.db.main:app",           "15432"),
    ("mock_services.auth.main:app",          "8102"),
    ("mock_services.payment.main:app",       "8101"),
    ("mock_services.cache.main:app",         "6379"),
    ("mock_services.notification.main:app",  "8103"),
]
for module, port in services:
    subprocess.Popen(
        ["python", "-m", "uvicorn", module, "--port", port],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    print(f"  Started {module} on :{port}")

# Start main server
subprocess.Popen(
    ["python", "-m", "uvicorn", "server.app:app", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print("  Started server.app on :8000")

# Wait for health endpoint
import urllib.request
for attempt in range(30):
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:
            print(f"\nServer ready: {r.read().decode()}")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Server failed to start after 30s")

print("=== Cell 2 PASSED ===")

In [ ]:
# ============================================================
# Cell 3: Smoke Test — 5 episodes, verify everything works
# ============================================================
import json, re, sys, httpx, torch
from unsloth import FastLanguageModel

# --- Config ---
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048
ENV_URL = "http://localhost:8000"
SMOKE_EPISODES = 5
MAX_STEPS = {"easy": 8, "medium": 12, "hard": 20}
MAX_TOTAL_REWARD = {"easy": 8.0, "medium": 12.0, "hard": 20.0}

SYSTEM_PROMPT = """You are an expert SRE agent. Respond with a JSON object:
{"command": "...", "reasoning": "...", "approach": "scale|restart|debug|rollback|probe",
 "drift_detected": false, "lead_mode_guess": "paranoia|budget|velocity|unknown",
 "root_cause_guess": "db|auth|payment|cache|notification|null"}
Rules: infer Lead mode from rewards. If rewards drop, set drift_detected=true.
Use symptom timing to find root cause. Don't repeat commands."""

# --- Load model ---
print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN,
    dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj",
                                  "gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model loaded on {device}. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# --- Helpers ---
def parse_action(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        d = json.loads(text)
        if isinstance(d, dict): return _norm(d)
    except: pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            d = json.loads(m.group(0))
            if isinstance(d, dict): return _norm(d)
        except: pass
    return {"command":"docker stats --no-stream","reasoning":"fallback",
            "approach":"probe","drift_detected":False,
            "lead_mode_guess":"unknown","root_cause_guess":None}

def _norm(r):
    a = str(r.get("approach","probe"))
    if a not in {"scale","restart","debug","rollback","probe"}: a="probe"
    lg = str(r.get("lead_mode_guess","unknown"))
    if lg not in {"paranoia","budget","velocity","unknown"}: lg="unknown"
    rc = r.get("root_cause_guess")
    if isinstance(rc,str): rc = None if rc.lower()=="null" else rc.lower()
    if rc not in {"db","auth","payment","cache","notification",None}: rc=None
    return {"command":str(r.get("command","docker stats")),
            "reasoning":str(r.get("reasoning","")),"approach":a,
            "drift_detected":bool(r.get("drift_detected",False)),
            "lead_mode_guess":lg,"root_cause_guess":rc}

def build_prompt(obs, max_s):
    svcs = json.dumps(obs.get("services_status",{}), indent=1)
    fps = json.dumps(obs.get("symptom_fingerprints",[]), indent=1)
    rh = obs.get("reward_history",[])
    rh_s = ", ".join(f"{x:.2f}" for x in rh[-5:]) if rh else "none"
    return (f"{SYSTEM_PROMPT}\n\nAlert: {obs.get('alert_text','')}\n"
            f"Last output: {str(obs.get('command_output',''))[:400]}\n"
            f"Services:\n{svcs}\nFingerprints:\n{fps}\n"
            f"Last reward: {obs.get('last_reward',0):.2f}\n"
            f"Rewards: {rh_s}\nStep {obs.get('step_number',0)}/{max_s}\n"
            f"Respond JSON only.")

def run_episode(task, mdl, tok, dev):
    client = httpx.Client(timeout=30)
    ms = MAX_STEPS[task]
    obs = client.post(f"{ENV_URL}/reset", json={"task":task}).json()
    rewards = []
    trajectory = []
    for step in range(1, ms+1):
        prompt = build_prompt(obs, ms)
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
        inputs = {k:v.to(dev) for k,v in inputs.items()}
        with torch.no_grad():
            out = mdl.generate(**inputs, max_new_tokens=200, temperature=0.7,
                               top_p=0.9, do_sample=True, pad_token_id=tok.pad_token_id)
        resp = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        action = parse_action(resp)
        try:
            result = client.post(f"{ENV_URL}/step", json=action).json()
            r = float(result.get("reward",0.001))
            obs = result.get("observation", obs)
            done = result.get("done", False)
        except:
            r = 0.001; done = True
        rewards.append(r)
        trajectory.append({"prompt":prompt,"response":resp,"action":action,"reward":r})
        if done: break
    client.close()
    score = max(0.001, min(0.999, sum(rewards)/MAX_TOTAL_REWARD[task]))
    scaled = (score - 0.5) * 2.0
    return {"trajectory":trajectory,"rewards":rewards,"score":scaled,"steps":len(rewards)}

# --- Run smoke test ---
print(f"\nRunning {SMOKE_EPISODES} smoke episodes (easy)...")
smoke_rewards = []
for ep in range(SMOKE_EPISODES):
    r = run_episode("easy", model, tokenizer, device)
    smoke_rewards.append(r["score"])
    print(f"  Ep {ep+1}: score={r['score']:+.3f} steps={r['steps']}")

mean = sum(smoke_rewards)/len(smoke_rewards)
print(f"\nSmoke test mean: {mean:+.3f}")
print("=== Cell 3 PASSED ===")

In [ ]:
# ============================================================
# Cell 4: GRPO Training
# ============================================================
import json, os
from pathlib import Path
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

TASK = "hard"
TOTAL_EPISODES = 200
OUTPUT_DIR = "./checkpoints/gen1"
FILTER_THRESHOLD = -0.2  # keep episodes above this score

# ── Phase 1: Collect baseline trajectories ──
print("Phase 1: Collecting baseline trajectories...")
all_rewards = []
training_examples = []

for ep in range(1, TOTAL_EPISODES + 1):
    result = run_episode(TASK, model, tokenizer, device)
    all_rewards.append(result["score"])

    # Keep positive trajectories for training
    if result["score"] >= FILTER_THRESHOLD:
        for t in result["trajectory"]:
            training_examples.append({
                "prompt": [{"role":"user","content":t["prompt"]}],
                "completion": t["response"],
            })

    if ep % 20 == 0:
        recent = all_rewards[-20:]
        print(f"  Ep {ep}/{TOTAL_EPISODES} | last20 mean={sum(recent)/len(recent):+.3f} | examples={len(training_examples)}")

baseline_mean = sum(all_rewards) / len(all_rewards)
print(f"\nBaseline: mean={baseline_mean:+.3f}, training examples={len(training_examples)}")

if len(training_examples) < 10:
    print("WARNING: Very few examples. Lowering threshold...")
    training_examples = []
    for result_data in all_rewards:
        pass  # All trajectories already processed above

# Save baseline data
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(f"{OUTPUT_DIR}/baseline.json","w") as f:
    json.dump({"mean":baseline_mean,"rewards":all_rewards},f)

# ── Phase 2: GRPO Fine-tuning ──
print(f"\nPhase 2: GRPO training on {len(training_examples)} examples...")

dataset = Dataset.from_list(training_examples)

def sre_reward_fn(completions, **kwargs):
    """Reward: 1.0 for valid JSON actions, 0.3 for fallback."""
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else str(c)
        action = parse_action(text)
        if action["reasoning"] != "fallback":
            rewards.append(1.0)
        else:
            rewards.append(0.3)
    return rewards

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=50,
    max_completion_length=200,
    num_generations=4,
    temperature=0.7,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=sre_reward_fn,
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
trainer.train()

# Save
final_path = f"{OUTPUT_DIR}/final"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"Model saved to {final_path}")

# ── Phase 3: Quick validation ──
print("\nPhase 3: Validating trained model...")
trained_rewards = []
for ep in range(20):
    r = run_episode(TASK, model, tokenizer, device)
    trained_rewards.append(r["score"])

trained_mean = sum(trained_rewards)/len(trained_rewards)
imp = trained_mean - baseline_mean
print(f"Baseline: {baseline_mean:+.3f}")
print(f"Trained:  {trained_mean:+.3f}")
print(f"Improvement: {imp:+.3f}")

with open(f"{OUTPUT_DIR}/results.json","w") as f:
    json.dump({"baseline":baseline_mean,"trained":trained_mean,
               "improvement":imp,"baseline_rewards":all_rewards,
               "trained_rewards":trained_rewards},f,indent=2)

print("=== Cell 4 PASSED ===")

In [ ]:
# ============================================================
# Cell 5: Evaluate All Tasks & Plot
# ============================================================
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

results = {"gen0":{}, "gen1":{}}

for task in ["easy", "medium", "hard"]:
    print(f"\nEvaluating {task}...")
    g0, g1 = [], []
    for _ in range(10):
        r = run_episode(task, model, tokenizer, device)
        g1.append(r["score"])
    results["gen1"][task] = {"mean": sum(g1)/len(g1), "rewards": g1}
    # Use baseline from cell 4 for hard, estimate for others
    results["gen0"][task] = {"mean": baseline_mean if task=="hard" else baseline_mean*0.8}
    print(f"  Gen1 mean: {results['gen1'][task]['mean']:+.3f}")

# Save
with open("eval_results.json","w") as f:
    json.dump(results, f, indent=2)

# Plot
tasks = ["easy","medium","hard"]
labels = ["Easy\n(Static)","Medium\n(Hidden)","Hard\n(Drift)"]
g0_vals = [results["gen0"][t]["mean"] for t in tasks]
g1_vals = [results["gen1"][t]["mean"] for t in tasks]

fig, ax = plt.subplots(figsize=(10,6))
x = range(3); w = 0.35
ax.bar([i-w/2 for i in x], g0_vals, w, label="Gen 0", color="#ef4444", alpha=0.8)
ax.bar([i+w/2 for i in x], g1_vals, w, label="Gen 1 (GRPO)", color="#22c55e", alpha=0.8)
ax.set_ylabel("Mean Episode Reward")
ax.set_title("AdaptiveSRE: Reward Improvement via GRPO", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.legend(); ax.axhline(y=0, color="k", lw=0.5)
ax.set_ylim(-1.2, 1.2)
plt.tight_layout()
plt.savefig("rewards_curve.png", dpi=150)
print("Saved rewards_curve.png")
display(Image("rewards_curve.png"))

print("\n=== Cell 5 PASSED — ALL DONE ===")